# 1 · The toy network

This series explains `DiscreteBN` one method at a time, so the whole team can trust it. Each method is built up from the simplest case, generalized step by step until it matches `bn.py`, then compared against the class.

| # | notebook | method(s) | trusts |
|---|---|---|---|
| 1 | `01_toy_network` | the toy network, its CPTs, the factorization | `Network`, `CPT` |
| 2 | `02_unnormalized` | `_unnormalized` | 1 |
| 3 | `03_posterior` | `_joint`, `posterior`, `log_likelihood`, `_predict_proba` | 1–2 |
| 4 | `04_counting` | `_complete_counts`, `_init_cpts` | 1 |
| 5 | `05_expected_counts` | `_expected_counts` (the E-step) | 1–4 |
| 6 | `06_em` | `_log_prior`, `fit` (the EM loop) | 1–5 |

Every notebook follows the same pattern: **one fixed row with numbers you can check by hand → loop → general → compare with `DiscreteBN`**. A ✅ means our version and the class agree.

In [ ]:
%load_ext autoreload
%autoreload 2

import itertools

import numpy as np
import pandas as pd
from toy import (
    CARDS,
    CO,
    DATA,
    EVIDENCE,
    NO2,
    S1,
    S4,
    STRUCTURE,
    TABLES,
    H,
    T,
    cpts,
    network,
    same,
    toy_bn,
)

from cse250a.modules.core import CPT

net = network()
bn = toy_bn()   # DiscreteBN with the hand-picked CPTs, not fitted

## The graph

```
    hour ──► CO ──► S1 ◄── T
      │       ╲
      └─► NO2 ─►S4
```

Same shape as Phase 2, shrunk: **two inputs** (hour, T), **two references** (CO, NO2), **two sensors**. S4 responds to NO2 (its nominal gas) *and* CO (a cross-sensitivity), so a high S4 could be either gas. That's the interesting part.

Every variable is **binary** (0 = low, 1 = high) so every probability can be checked by hand. Hour here is just night/day; the real one has 24 states.

In [ ]:
print("inputs   (no CPT, always observed):", net.inputs)
print("modeled  (each has a CPT):         ", net.nodes)
print()
for node in net.nodes:
    print(f"{node!s:>14}  family {tuple(map(str, net.family(node)))}  →  CPT shape {net.shape(node)}")

## The CPTs

A CPT's axes are `(*parents, node)`, so `TABLES[S1][co, t]` is the distribution of S1 when CO = `co` and T = `t`. `to_frame()` shows one row per parent combination; every row sums to 1.

In [ ]:
for node in net.nodes:
    print(f"P({node} | {', '.join(map(str, STRUCTURE[node]))})")
    display(cpts()[node].to_frame().round(2))

In [ ]:
for node, table in TABLES.items():
    same(f"rows of P({node} | parents) sum to 1", table.sum(axis=-1), np.ones(table.shape[:-1]))

## The data

Eight hand-made rows, each covering a case the methods must handle. NaN means missing.

| row | what's missing | why it matters |
|---|---|---|
| 0, 1 | nothing | the plain counting case |
| 2 | CO | one hidden reference |
| 3 | NO2 | the other one |
| 4 | CO and NO2 | exactly the situation at **prediction** time: references hidden, sensors seen |
| 5, 6 | a sensor | sensors go missing too |
| 7 | CO, NO2, S1 | three hidden at once |

In [ ]:
DATA

## The one idea everything rests on: the factorization

A Bayesian network says the joint distribution is **the product of every node's CPT entry**:

$$P(\text{CO}, \text{NO2}, S_1, S_4 \mid \text{hour}, T) = P(\text{CO}\mid \text{hour})\; P(\text{NO2}\mid \text{hour})\; P(S_1 \mid \text{CO}, T)\; P(S_4 \mid \text{NO2}, \text{CO})$$

Inputs have no CPT: we *condition* on them. Here it is for row 0 (hour=1, T=0, CO=1, NO2=1, S1=1, S4=1), one factor at a time:

In [ ]:
f_co  = TABLES[CO][1, 1]       # P(CO=1  | hour=1)
f_no2 = TABLES[NO2][1, 1]      # P(NO2=1 | hour=1)
f_s1  = TABLES[S1][1, 0, 1]    # P(S1=1  | CO=1, T=0)
f_s4  = TABLES[S4][1, 1, 1]    # P(S4=1  | NO2=1, CO=1)
print(f"{f_co} × {f_no2} × {f_s1} × {f_s4} = {f_co * f_no2 * f_s1 * f_s4:.4f}")

**Check that it really is a distribution.** For each fixed (hour, T), summing the product over all $2^4 = 16$ assignments of the modeled nodes must give exactly 1. That's what "conditioned on the inputs" means.

In [ ]:
for h, t in itertools.product(range(2), range(2)):
    total = sum(
        TABLES[CO][h, co] * TABLES[NO2][h, no2] * TABLES[S1][co, t, s1] * TABLES[S4][no2, co, s4]
        for co, no2, s1, s4 in itertools.product(range(2), repeat=4)
    )
    print(f"hour={h}, T={t}:  Σ = {total:.12f}")

**Everything in `DiscreteBN` is this product, used three ways:**

- **Inference** (notebooks 2–3): fix the observed cells, sum the product over the missing ones.
- **Counting** (notebook 4): with nothing missing, learning a CPT is just counting.
- **EM** (notebooks 5–6): with cells missing, count *fractionally*, weighting each possibility by its posterior.